---

## **Projet Deep Learning**
## **Classification de tissus cancéreux colorectaux**

---


---

**Ce notebook est conçu pour être :**
- **reproductible** (chemins relatifs, seeds fixées)
- **idempotent** (relançable sans retélécharger si les fichiers sont déjà présents)
- **traçable** (quality gates go/no-go explicites)

---


---

# PARTIE 3 : CNN FROM SCRATCH

---


---

### Objectif

Construire un CNN pour dépasser les limites du MLP. Le MLP de la partie 2 plafonnait à 63% car il traitait chaque pixel indépendamment. Le CNN va balayer l'image avec des filtres qui détectent les textures, les contours et les formes.

### Progression du projet
- **Partie 2 (MLP)** : 63.08% test accuracy
- **Partie 3 (CNN)** : exploiter la structure spatiale des images

### Consignes du sujet
- Architecture : ≥ 3 blocs convolutifs + batch normalization + dropout
- Data augmentation justifiée
- Seuil : ≥ 75% test accuracy
- Comparer avec vs sans augmentation (même modèle, mêmes hyperparamètres, 40 époques)
- **Q3.1** : Sans augmentation, 40 époques — à quelle époque le gap train/val accuracy dépasse 15 points ?
- **Q3.2** : Une augmentation qui exploite l'absence d'orientation canonique en histologie + une augmentation nuisible
- **Q3.3** : Nombre total de paramètres + calcul manuel de la 1ère couche conv

---


---

### Choix d'architecture et d'hyperparamètres CNN

**Lien avec le Lab 2 :**
- Architecture inspirée des exercices 1 à 3 du Lab 2 (VGG-like + régularisation)

**Architecture CNN (3 blocs convolutifs) :**
- Chaque bloc : Conv2d → BatchNorm → ReLU → Conv2d → BatchNorm → ReLU → MaxPool → Dropout
- Filtres : 32 → 64 → 128 (progressif)
- Filtres 3×3 avec padding=1

**Batch Normalization :** stabilise et accélère l'entraînement

**Dropout (0.25 → 0.35 → 0.5) :** régularisation progressive

**Data augmentation :** géométrie pure (flips + rotation 90°) — pas de modification de couleur ni de netteté à 28×28

**CrossEntropyLoss + Adam (lr=0.001) :** cohérence avec la partie 2

---


In [ ]:
# Lien avec le notebook 1 (EDA) — imports
# Version 1.0 — 20 mars 2026

import sys
import os
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import medmnist
from medmnist import PathMNIST
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
print("Imports OK")


In [ ]:
# Versions
print(f"Python   : {sys.version.split()[0]}")
print(f"PyTorch  : {torch.__version__}")
print(f"MedMNIST : {medmnist.__version__}")
print(f"NumPy    : {np.__version__}")


In [ ]:
# Reproductibilité
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"cuDNN determ. : {torch.backends.cudnn.deterministic if torch.cuda.is_available() else 'N/A'}")


In [ ]:
# Constantes et dataset
DATA_DIR = os.path.join(".", "data")
NORM_MEAN = [0.7405, 0.5330, 0.7058]
NORM_STD  = [0.1237, 0.1768, 0.1244]
N_CLASSES = 9

train_dataset = PathMNIST(split='train', download=False, root=DATA_DIR)
val_dataset   = PathMNIST(split='val',   download=False, root=DATA_DIR)
test_dataset  = PathMNIST(split='test',  download=False, root=DATA_DIR)

info = train_dataset.info
labels_names = info['label']
CLASS_NAMES = list(labels_names.values())

notebook_start_time = time.time()

print(f"NORM_MEAN : {NORM_MEAN}")
print(f"NORM_STD  : {NORM_STD}")
print(f"Train : {len(train_dataset)} | Val : {len(val_dataset)} | Test : {len(test_dataset)}")
print("✓ Lien avec notebook 1 établi")


In [ ]:
# Fonction d'entraînement
def train_model(model, train_loader, val_loader, epochs=40, learning_rate=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    best_val_loss = float('inf')
    best_model_state = None
    best_epoch = 0
    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for images, labels_batch in train_loader:
            images = images.to(device)
            labels_batch = labels_batch.squeeze().long().to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
            train_correct += (outputs.argmax(1) == labels_batch).sum().item()
            train_total += images.size(0)
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for images, labels_batch in val_loader:
                images = images.to(device)
                labels_batch = labels_batch.squeeze().long().to(device)
                outputs = model(images)
                loss = criterion(outputs, labels_batch)
                val_loss += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels_batch).sum().item()
                val_total += images.size(0)
        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        history['loss'].append(epoch_train_loss)
        history['accuracy'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_accuracy'].append(epoch_val_acc)
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
        print(f"Epoch {epoch+1:>3}/{epochs} | "
              f"Train loss: {epoch_train_loss:.6f} acc: {epoch_train_acc:.6f} | "
              f"Val loss: {epoch_val_loss:.6f} acc: {epoch_val_acc:.6f}"
              f"{' <- best' if epoch + 1 == best_epoch else ''}")
    model.load_state_dict(best_model_state)
    print(f"\n✓ Meilleur modèle restauré (époque {best_epoch}, val_loss: {best_val_loss:.6f})")
    return history

print("✓ Fonction train_model chargée")


In [ ]:
# Preprocessing CNN
cnn_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

cnn_transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

train_cnn     = PathMNIST(split='train', transform=cnn_transform, download=False, root=DATA_DIR)
train_cnn_aug = PathMNIST(split='train', transform=cnn_transform_aug, download=False, root=DATA_DIR)
val_cnn       = PathMNIST(split='val',   transform=cnn_transform, download=False, root=DATA_DIR)
test_cnn      = PathMNIST(split='test',  transform=cnn_transform, download=False, root=DATA_DIR)

BATCH_SIZE = 64
train_loader_cnn     = DataLoader(train_cnn, batch_size=BATCH_SIZE, shuffle=True)
train_loader_cnn_aug = DataLoader(train_cnn_aug, batch_size=BATCH_SIZE, shuffle=True)
val_loader_cnn       = DataLoader(val_cnn, batch_size=BATCH_SIZE, shuffle=False)
test_loader_cnn      = DataLoader(test_cnn, batch_size=BATCH_SIZE, shuffle=False)

sample_img, _ = train_cnn[0]
assert sample_img.shape == (3, 28, 28)
print(f"Image CNN : shape={sample_img.shape}")
print("✓ Preprocessing CNN terminé")


In [ ]:
# Modèle CNN + Q3.3
def create_cnn(n_classes=9):
    model = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
        nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
        nn.MaxPool2d(2, 2), nn.Dropout(0.25),
        nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
        nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
        nn.MaxPool2d(2, 2), nn.Dropout(0.35),
        nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        nn.MaxPool2d(2, 2), nn.Dropout(0.5),
        nn.Flatten(),
        nn.Linear(128 * 3 * 3, 128), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(128, n_classes)
    )
    return model

print("=== Q3.3 ===")
print(f"  1ère couche : (3×3×3 + 1) × 32 = {(3*3*3+1)*32} paramètres")

torch.manual_seed(SEED)
cnn_model = create_cnn()
cnn_model = cnn_model.to(device)
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f"  Nb total paramètres : {total_params:,}")
print("✓ Modèle CNN créé")


---

## Expérience 1 : CNN sans augmentation (consigne Q3.1)

---


In [ ]:
# Entraînement CNN SANS augmentation — 40 époques
start_time = time.time()
print("=== Entraînement CNN SANS augmentation (40 époques) ===")
history_cnn_no_aug = train_model(cnn_model, train_loader_cnn, val_loader_cnn, epochs=40, learning_rate=0.001)
train_time_no_aug = time.time() - start_time
print(f"\nTemps : {train_time_no_aug:.1f}s ({train_time_no_aug/60:.1f} min)")


In [ ]:
# Courbes + Q3.1 gap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history_cnn_no_aug['accuracy']) + 1)
ax1.plot(epochs_range, history_cnn_no_aug['accuracy'], label='Train')
ax1.plot(epochs_range, history_cnn_no_aug['val_accuracy'], label='Val')
ax1.set_xlabel('Époque'); ax1.set_ylabel('Accuracy'); ax1.set_title('Accuracy par époque'); ax1.legend()
ax2.plot(epochs_range, history_cnn_no_aug['loss'], label='Train')
ax2.plot(epochs_range, history_cnn_no_aug['val_loss'], label='Val')
ax2.set_xlabel('Époque'); ax2.set_ylabel('Loss'); ax2.set_title('Loss par époque'); ax2.legend()
plt.suptitle("CNN sans augmentation (40 époques)", fontsize=14)
plt.tight_layout(); plt.show()

print("\n=== Q3.1 — Gap train/val accuracy ===")
max_gap = -999
for e in range(len(history_cnn_no_aug['accuracy'])):
    gap = (history_cnn_no_aug['accuracy'][e] - history_cnn_no_aug['val_accuracy'][e]) * 100
    if gap > max_gap: max_gap = gap
    if e < 5 or e >= 35 or abs(gap) > 5:
        print(f"  Époque {e+1:>3} | train: {history_cnn_no_aug['accuracy'][e]:.6f} | val: {history_cnn_no_aug['val_accuracy'][e]:.6f} | gap: {gap:+.2f} pts")
print(f"\nGap maximal : {max_gap:.2f} pts")
print("Le gap ne dépasse jamais 15 points." if max_gap < 15 else "Le gap dépasse 15 points.")


In [ ]:
# Test set — sans augmentation
cnn_model.eval()
test_correct, test_total = 0, 0
all_preds_cnn = []
all_labels_cnn = []
with torch.no_grad():
    for images, labels_batch in test_loader_cnn:
        images = images.to(device)
        labels_batch = labels_batch.squeeze().long().to(device)
        outputs = cnn_model(images)
        preds = outputs.argmax(1)
        test_correct += (preds == labels_batch).sum().item()
        test_total += images.size(0)
        all_preds_cnn.extend(preds.cpu().numpy())
        all_labels_cnn.extend(labels_batch.cpu().numpy())
test_accuracy_cnn = test_correct / test_total
print(f"Test accuracy sans aug : {test_accuracy_cnn:.6f}")
print(f"Seuil >= 75% -> {'✓ ATTEINT' if test_accuracy_cnn >= 0.75 else '✗ NON ATTEINT'}")


In [ ]:
# Matrice de confusion + report — sans augmentation
cm_cnn = confusion_matrix(all_labels_cnn, all_preds_cnn)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
ax.set_xlabel('Prédit'); ax.set_ylabel('Vrai')
ax.set_title('Matrice de confusion — CNN sans augmentation (test set)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

cm_off = cm_cnn.copy()
np.fill_diagonal(cm_off, 0)
max_idx = np.unravel_index(cm_off.argmax(), cm_off.shape)
print(f"\nPlus grande confusion : {cm_off[max_idx]} images de '{CLASS_NAMES[max_idx[0]]}' classées comme '{CLASS_NAMES[max_idx[1]]}'")
print("\n=== Classification report ===")
print(classification_report(all_labels_cnn, all_preds_cnn, target_names=CLASS_NAMES, digits=4))


---

### Résultats CNN sans augmentation

*(À compléter avec les chiffres après exécution)*

---


---

## Expérience 2 : CNN avec augmentation (comparaison, 40 époques)

Augmentation retenue : géométrie pure (flips + rotation 90°).

---


In [ ]:
# Entraînement CNN AVEC augmentation — 40 époques
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)
cnn_model_aug = create_cnn()
cnn_model_aug = cnn_model_aug.to(device)

start_time = time.time()
print("=== Entraînement CNN AVEC augmentation (40 époques) ===")
history_cnn_aug = train_model(cnn_model_aug, train_loader_cnn_aug, val_loader_cnn, epochs=40, learning_rate=0.001)
train_time_aug = time.time() - start_time
print(f"\nTemps : {train_time_aug:.1f}s ({train_time_aug/60:.1f} min)")


In [ ]:
# Courbes — avec augmentation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history_cnn_aug['accuracy']) + 1)
ax1.plot(epochs_range, history_cnn_aug['accuracy'], label='Train')
ax1.plot(epochs_range, history_cnn_aug['val_accuracy'], label='Val')
ax1.set_xlabel('Époque'); ax1.set_ylabel('Accuracy'); ax1.set_title('Accuracy par époque'); ax1.legend()
ax2.plot(epochs_range, history_cnn_aug['loss'], label='Train')
ax2.plot(epochs_range, history_cnn_aug['val_loss'], label='Val')
ax2.set_xlabel('Époque'); ax2.set_ylabel('Loss'); ax2.set_title('Loss par époque'); ax2.legend()
plt.suptitle("CNN avec augmentation (40 époques)", fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# Test set — avec augmentation
cnn_model_aug.eval()
test_correct, test_total = 0, 0
all_preds_cnn_aug = []
all_labels_cnn_aug = []
with torch.no_grad():
    for images, labels_batch in test_loader_cnn:
        images = images.to(device)
        labels_batch = labels_batch.squeeze().long().to(device)
        outputs = cnn_model_aug(images)
        preds = outputs.argmax(1)
        test_correct += (preds == labels_batch).sum().item()
        test_total += images.size(0)
        all_preds_cnn_aug.extend(preds.cpu().numpy())
        all_labels_cnn_aug.extend(labels_batch.cpu().numpy())
test_accuracy_cnn_aug = test_correct / test_total
print(f"Test accuracy avec aug : {test_accuracy_cnn_aug:.6f}")
print(f"Seuil >= 75% -> {'✓ ATTEINT' if test_accuracy_cnn_aug >= 0.75 else '✗ NON ATTEINT'}")


In [ ]:
# Matrice de confusion + report — avec augmentation
cm_cnn_aug = confusion_matrix(all_labels_cnn_aug, all_preds_cnn_aug)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_cnn_aug, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
ax.set_xlabel('Prédit'); ax.set_ylabel('Vrai')
ax.set_title('Matrice de confusion — CNN avec augmentation (test set)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

cm_aug_off = cm_cnn_aug.copy()
np.fill_diagonal(cm_aug_off, 0)
max_idx = np.unravel_index(cm_aug_off.argmax(), cm_aug_off.shape)
print(f"\nPlus grande confusion : {cm_aug_off[max_idx]} images de '{CLASS_NAMES[max_idx[0]]}' classées comme '{CLASS_NAMES[max_idx[1]]}'")
print("\n=== Classification report ===")
print(classification_report(all_labels_cnn_aug, all_preds_cnn_aug, target_names=CLASS_NAMES, digits=4))


In [ ]:
# Tableau comparatif
print("=" * 75)
print("COMPARAISON CNN : SANS vs AVEC augmentation (40 époques)")
print("=" * 75)
print(f"{'Métrique':<30s} {'Sans aug':>15s} {'Avec aug':>15s} {'Diff':>12s}")
print("-" * 75)
print(f"{'Val accuracy (best)':<30s} {max(history_cnn_no_aug['val_accuracy']):>15.6f} {max(history_cnn_aug['val_accuracy']):>15.6f} {(max(history_cnn_aug['val_accuracy'])-max(history_cnn_no_aug['val_accuracy']))*100:>+12.2f} pts")
print(f"{'Test accuracy':<30s} {test_accuracy_cnn:>15.6f} {test_accuracy_cnn_aug:>15.6f} {(test_accuracy_cnn_aug-test_accuracy_cnn)*100:>+12.2f} pts")
print(f"{'Temps':<30s} {train_time_no_aug:>14.1f}s {train_time_aug:>14.1f}s")


---

### Q3.1 — Gap train/val accuracy sur 40 époques sans augmentation

*(Voir la cellule 15 pour les chiffres exacts. Le gap dépasse-t-il 15 points ?)*

### Q3.2 — Augmentations pertinentes et nuisibles

**Augmentation exploitant l'absence d'orientation canonique :**
RandomHorizontalFlip + RandomVerticalFlip + RandomRotation(90°) — en histologie, les lames n'ont pas d'orientation fixe. Un tissu retourné ou pivoté reste le même tissu.

**Augmentation nuisible :**
- RandomGrayscale à forte probabilité — supprimerait la coloration H&E, signal discriminant principal
- RandomCrop agressif — à 28×28, couper l'image perdrait trop d'information
- ColorJitter ou GaussianBlur — détruisent les textures fines à cette résolution

### Q3.3 — Nombre de paramètres

*(Voir la cellule 12 pour le calcul manuel et le total)*

---


---

## BONUS — Itérations sur les augmentations (démarche expérimentale)

Les expériences ci-dessous documentent notre démarche itérative. Elles sont hors consignes mais enrichissent l'analyse.

| Version | Augmentations | Époques | Test acc | Observation |
|---------|--------------|---------|----------|-------------|
| Sans aug | Aucune | 40 | 92.38% | Meilleur test accuracy |
| v1 | Flips + Rot90 + ColorJitter 0.1 + Grayscale 10% | 40 | 88.86% | ColorJitter et Grayscale nuisent |
| v2 | Flips + Rot90 + ColorJitter 0.05 + GaussianBlur | 50 | 83.86% | GaussianBlur détruit les textures |
| v3 | Flips + Rot90 uniquement | 70 | 85.78% | Géométrie pure, domain shift reste |

### Ce que les itérations nous apprennent

1. Les augmentations de couleur nuisent — les tissus conjonctifs se distinguent par de subtiles nuances H&E
2. L'augmentation géométrique est neutre sur le val mais dégrade le test — ne résout pas le domain shift
3. Plus d'époques = plus de mémorisation de l'hôpital A
4. Le meilleur modèle est le plus simple — 40 époques, pas d'augmentation

---


In [ ]:
# Sauvegarde du meilleur modèle CNN
torch.save(cnn_model.state_dict(), os.path.join(DATA_DIR, 'cnn_model.pth'))

cnn_results = {
    'model_name': 'CNN',
    'n_params': sum(p.numel() for p in cnn_model.parameters()),
    'test_accuracy_no_aug': test_accuracy_cnn,
    'test_accuracy_aug': test_accuracy_cnn_aug,
}
print(f"Modèle CNN sauvegardé")
print(f"Test acc sans aug : {test_accuracy_cnn:.6f}")
print(f"Test acc avec aug : {test_accuracy_cnn_aug:.6f}")
print(f"Nb paramètres    : {cnn_results['n_params']:,}")
print("✓ Sauvegarde terminée")


---

## Bilan Partie 3 — CNN from scratch

*(À compléter avec les chiffres après exécution)*

---


In [ ]:
# Temps total
notebook_total_time = time.time() - notebook_start_time
print(f"Temps total du notebook : {notebook_total_time:.1f}s ({notebook_total_time/60:.1f} min)")
